# Weather Agent

This notebook simulates weather conditions in Copenhagen by alternating between rain and sun periods.

**Behavior:**
- Publishes weather state to MQTT topic: `simulated_city/weather/rain`
- Alternates between:
  - Rain: 10 seconds
  - Sun: 20 seconds
- Runs indefinitely until stopped

**MQTT Message Format:**
```json
{
  "raining": true/false,
  "duration_sec": 10 or 20
}
```

**To verify this agent works:**
1. Ensure local MQTT broker is running: `mosquitto -v`
2. In another terminal, subscribe: `mosquitto_sub -h localhost -t "simulated_city/weather/rain"`
3. Run all cells in this notebook
4. Watch the messages appear in the subscriber terminal

In [1]:
# Import required libraries
import time
import json
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher

In [2]:
# Load configuration and connect to MQTT broker
cfg = load_config()

# Create MQTT connector and publisher
connector = MqttConnector(cfg.mqtt, client_id_suffix="weather")
connector.connect()

# Wait for connection to establish
if not connector.wait_for_connection(timeout=10.0):
    raise ConnectionError("Failed to connect to MQTT broker")

publisher = MqttPublisher(connector)
print("Connected to MQTT broker successfully")
print(f"Publishing to topic: simulated_city/weather/rain")

Connected to MQTT broker successfully
Publishing to topic: simulated_city/weather/rain


In [3]:
# Main weather simulation loop
# Alternates between rain (10s) and sun (20s)
RAIN_DURATION = 10  # seconds
SUN_DURATION = 20   # seconds
TOPIC = "simulated_city/weather/rain"

print("Starting weather simulation...")
print("Press Interrupt (■) button to stop\n")

try:
    cycle = 0
    while True:
        cycle += 1
        
        # Sun phase
        sun_message = {
            "raining": False,
            "duration_sec": SUN_DURATION
        }
        publisher.publish_json(TOPIC, json.dumps(sun_message), qos=0, retain=False)
        print(f"[Cycle {cycle}] ☀️  Sun - published: {sun_message}")
        time.sleep(SUN_DURATION)
        
        # Rain phase
        rain_message = {
            "raining": True,
            "duration_sec": RAIN_DURATION
        }
        publisher.publish_json(TOPIC, json.dumps(rain_message), qos=0, retain=False)
        print(f"[Cycle {cycle}] 🌧️  Rain - published: {rain_message}")
        time.sleep(RAIN_DURATION)
        
except KeyboardInterrupt:
    print("\n⚠️  Simulation stopped by user")

Starting weather simulation...
Press Interrupt (■) button to stop

[Cycle 1] ☀️  Sun - published: {'raining': False, 'duration_sec': 20}
[Cycle 1] 🌧️  Rain - published: {'raining': True, 'duration_sec': 10}
[Cycle 2] ☀️  Sun - published: {'raining': False, 'duration_sec': 20}
[Cycle 2] 🌧️  Rain - published: {'raining': True, 'duration_sec': 10}
[Cycle 3] ☀️  Sun - published: {'raining': False, 'duration_sec': 20}

⚠️  Simulation stopped by user


In [5]:
# Graceful shutdown - disconnect from MQTT broker
connector.disconnect()
print("Disconnected from MQTT broker")

Disconnected from MQTT broker
